# APS Preview / Results Extraction

This notebook is a simple standalone copy of the Manual APS `APS Preview / Results` plotting workflow.

It does not import `ca_app` or any app settings/classes. Edit the local functions below directly while testing APS fitting and plotting.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import csv
import re

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import shgo
from scipy.signal import savgol_filter
from scipy.special import erf

## Parameters

Add your APS04 file paths here. The fitting defaults match the APS GUI energy-window defaults.

In [ ]:
# Example Windows path:
# aps_paths = [r"C:\\path\\to\\sample_APS.csv"]
aps_paths = []

# APS fitting parameters.
fit_lower_ev = 5.45
fit_upper_ev = 6.0
fit_points = 7

# Signal power: None reads APS DAT metadata when present, otherwise defaults to "1/3".
# You can force "1/2" or "1/3".
power = None

# Optional DOS calculation parameters. DOS is not plotted in this APS Preview / Results notebook,
# but these are here if you want to inspect or adjust calculated DOS later.
calculate_dos = True
dos_window_length = 7
dos_polyorder = 3
dos_scale = 4.122623
dos_y0 = 0.1

## Local APS Data Loading

The APS04 loader uses the same column convention as the app: energy from column 3, APS 1/2 signal from column 7, and APS 1/3 signal from column 8.

In [ ]:
class ApsNotebookError(ValueError):
    pass


@dataclass
class ApsMetadata:
    values: dict[str, str]

    @property
    def has_fit_window(self):
        return self.e_low_ev is not None and self.e_high_ev is not None

    @property
    def e_low_ev(self):
        return self._mev_value("ELo")

    @property
    def e_high_ev(self):
        return self._mev_value("EHi")

    @property
    def wf_ev(self):
        return self._mev_value("WF")

    @property
    def power(self):
        return self.values.get("Power")

    @property
    def suggested_power(self):
        if self.power in ("1/2", "1/3"):
            return self.power
        return None

    def _mev_value(self, key):
        value = self._float_value(key)
        if value is None:
            return None
        return value / 1000.0

    def _float_value(self, key):
        try:
            return float(self.values[key])
        except (KeyError, TypeError, ValueError):
            return None


@dataclass
class ApsData:
    name: str
    energy: np.ndarray
    pes_raw: np.ndarray
    power: str
    metadata: ApsMetadata | None = None

    @property
    def sqrt(self):
        return self.power == "1/2"

    @property
    def is_analyzed(self):
        return hasattr(self, "homo")

    @property
    def has_baseline(self):
        return hasattr(self, "baseline")

    @property
    def has_cutoff(self):
        return hasattr(self, "cutoff_energy")

    @property
    def pes(self):
        if self.has_baseline:
            return self.pes_raw - self.baseline
        return self.pes_raw

    @property
    def pes_base(self):
        if not self.has_baseline:
            find_baseline(self)
        return self.pes_raw - self.baseline


def read_csv_rows(path):
    with open(path, "r", newline="", encoding="utf-8-sig") as handle:
        return list(csv.reader(handle))


def parse_aps_metadata(rows):
    values = {}
    for row in rows:
        for cell in row:
            for key, value in re.findall(r"\b([A-Za-z][A-Za-z0-9_]*)\s*=\s*([^,\s]+)", cell):
                values[key] = value
    if not values:
        return None
    return ApsMetadata(values)


def choose_power(metadata, requested_power=None):
    if requested_power in ("1/2", "1/3"):
        return requested_power
    if requested_power is not None:
        raise ApsNotebookError('power must be None, "1/2", or "1/3".')
    if metadata is not None and metadata.suggested_power is not None:
        return metadata.suggested_power
    return "1/3"


def load_aps_file(path, requested_power=None):
    path = Path(path).expanduser()
    rows = read_csv_rows(path)
    if not rows:
        raise ApsNotebookError(f"No data found in {path}.")

    metadata = parse_aps_metadata(rows)
    selected_power = choose_power(metadata, requested_power)
    signal_index = 6 if selected_power == "1/2" else 7
    body = rows[1:]

    stop_index = len(body)
    for index, row in enumerate(body):
        try:
            if "WF" not in row[0] and float(row[3]) < 1e4:
                continue
            stop_index = index
            break
        except (IndexError, ValueError) as exc:
            raise ApsNotebookError(f"{path} does not have the expected APS04 format.") from exc

    values = []
    for row in body[:stop_index]:
        try:
            values.append((float(row[2]), float(row[signal_index])))
        except (IndexError, ValueError) as exc:
            raise ApsNotebookError(f"{path} does not contain numeric APS columns 2 and {signal_index}.") from exc

    if not values:
        raise ApsNotebookError(f"No APS data found in {path}.")

    array = np.asarray(values, dtype=float)
    return ApsData(path.name, array[:, 0], array[:, 1], selected_power, metadata=metadata)


def load_aps_files(paths, requested_power=None):
    return [load_aps_file(path, requested_power=requested_power) for path in paths if str(path).strip()]

## Local APS Fitting

This is the editable fitting logic used by both figures. It fits all candidate line segments inside the selected energy window and keeps the segment with the smallest HOMO uncertainty.

In [ ]:
def gaussian(x, c, center):
    return np.exp(-((x - center) ** 2) / 2 / c**2) / c / (2 * np.pi) ** 0.5


def mofun(x, c, mo_energy):
    return np.sum([gaussian(x, c, center) for center in mo_energy], axis=0)


def find_baseline(item, baseline_bounds=(1.0, 10.0)):
    result = shgo(lambda x: -mofun(x, 0.3, item.pes_raw), [baseline_bounds], iters=2)
    if len(result.x) == 0:
        raise ApsNotebookError(f"Could not find baseline for {item.name}.")
    baseline = float(result.x[0])
    lower, upper = baseline_bounds
    if np.isclose(baseline, lower) or np.isclose(baseline, upper):
        raise ApsNotebookError(f"Found baseline is on the boundary ({baseline:.6g}) for {item.name}.")
    item.baseline = baseline
    return baseline


def fit_aps_energy_window(item, lower_ev, upper_ev, points=7):
    if points <= 2:
        raise ApsNotebookError("Linear fit must contain more than 2 points.")

    lower = min(float(lower_ev), float(upper_ev))
    upper = max(float(lower_ev), float(upper_ev))
    indexes = np.where((item.energy >= lower) & (item.energy <= upper))[0]
    if len(indexes) <= points:
        raise ApsNotebookError(f"Energy fitting window is too narrow for {item.name}.")

    start = int(indexes[0])
    stop = int(indexes[-1]) + 1
    item.fit_lower_bound = lower
    item.fit_upper_bound = upper
    item.fit_candidate_start_index = start
    item.fit_candidate_stop_index = stop
    item.std_homo = np.inf

    for i in range(start, stop):
        for j in range(i + int(points), stop):
            x = item.energy[i:j]
            y = item.pes_base[i:j]
            fit = np.polyfit(x, y, 1)
            if np.isclose(fit[0], 0.0):
                continue
            x_intercept = -fit[1] / fit[0]
            sig_square = ((np.polyval(fit, x) - y) ** 2).sum() / (j - i - 2)
            design = np.concatenate(([x - x_intercept], [np.repeat(-fit[0], j - i)]), 0)
            try:
                [[_, _], [_, var_homo]] = np.linalg.inv(design.dot(design.T)) * sig_square
            except np.linalg.LinAlgError:
                continue
            std_homo = var_homo**0.5
            if std_homo < item.std_homo:
                item.lin_start_index = i
                item.lin_stop_index = j
                item.lin_par = fit
                item.std_homo = std_homo
                item.homo = float(x_intercept)

    if item.std_homo == np.inf:
        raise ApsNotebookError(f"Fitting failed for {item.name}. Rechoose fitting conditions.")
    return item


def find_cutoff(item):
    try:
        index = next(len(item.pes_base) - i for i, value in enumerate(item.pes_base[::-1]) if value < 0)
    except StopIteration as exc:
        raise ApsNotebookError(f"Baseline was not correct for {item.name}; cutoff could not be found.") from exc
    item.cutoff_index = index
    item.cutoff_energy = float(item.energy[index])
    return index, item.cutoff_energy


def erfsmooth(x, scale, cutoff, y0):
    if y0 < 0:
        raise ApsNotebookError("DOS smoothing y0 must be >= 0.")
    return erf((x - cutoff) * scale) * (1 - y0) / 2 + (1 + y0) / 2


def smooth_dos(item, window_length=7, polyorder=3, scale=4.122623, y0=0.1):
    if window_length <= polyorder:
        raise ApsNotebookError("DOS smoothing window length must be greater than polynomial order.")
    if window_length % 2 == 0:
        raise ApsNotebookError("DOS smoothing window length must be odd.")
    if window_length > len(item.energy):
        raise ApsNotebookError(f"DOS smoothing window is longer than data length for {item.name}.")
    if not item.has_cutoff:
        find_cutoff(item)
    raw = np.gradient(item.pes, item.energy)
    cutoff = item.cutoff_energy + 0.053547
    item.dos_raw = raw
    item.dos = savgol_filter(raw * erfsmooth(item.energy, scale, cutoff, y0), window_length, polyorder)
    return item.dos


def analyze_aps_items(items):
    for item in items:
        fit_aps_energy_window(item, fit_lower_ev, fit_upper_ev, fit_points)
        if calculate_dos:
            smooth_dos(item, dos_window_length, dos_polyorder, dos_scale, dos_y0)
    return items

## Editable Plotting Helpers

In [ ]:
def aps_y_label(item_power):
    if item_power == "1/2":
        return "Photoemission$^{1/2}$  (arb.unit)"
    return "Photoemission$^{1/3}$  (arb. unit)"


def apply_aps_axis_style(ax):
    ax.grid(True, which="both", axis="x")
    ax.axhline(y=0, color="k", linestyle="--", linewidth=0.8)
    ax.autoscale(enable=True, axis="both", tight=True)
    ax.set_ylim(bottom=-0.5)


def show_labeled_legend(ax):
    handles, labels = ax.get_legend_handles_labels()
    pairs = [(handle, label) for handle, label in zip(handles, labels) if label and not label.startswith("_")]
    if pairs:
        ax.legend(*zip(*pairs), fontsize=8)


def shade_aps_fit_region(ax, lower, upper):
    ax.axvspan(lower, upper, color="tab:orange", alpha=0.18)


def plot_dat_reference_markers(ax, item):
    metadata = item.metadata
    if metadata is None:
        return
    if metadata.wf_ev is not None and metadata.wf_ev > 0:
        ax.axvline(metadata.wf_ev, color="tab:green", linestyle=":", linewidth=1.0, label=f"{item.name} DAT WF")


def plot_aps_fit(ax, item):
    if not item.is_analyzed:
        return
    x = item.energy[item.lin_start_index : item.lin_stop_index]
    y = item.lin_par[0] * x + item.lin_par[1]
    ax.plot(x, y, "--", linewidth=1.6, label=f"{item.name} fit")
    ax.axvspan(
        float(item.energy[item.lin_start_index]),
        float(item.energy[item.lin_stop_index - 1]),
        color="tab:blue",
        alpha=0.08,
        label="actual fitted segment",
    )
    ax.text(
        0.02,
        0.86,
        f"{item.name} used: {item.energy[item.lin_start_index]:.3f}-{item.energy[item.lin_stop_index - 1]:.3f} eV",
        transform=ax.transAxes,
        fontsize=8,
        bbox={"facecolor": "white", "alpha": 0.65, "edgecolor": "0.8"},
    )

In [ ]:
def plot_aps_preview_axis(ax, items):
    if not items:
        ax.text(0.5, 0.5, "Add APS file paths in the parameter cell.", ha="center", va="center", transform=ax.transAxes)
        item_power = "1/3"
    else:
        item_power = items[0].power
        for item in items:
            try:
                fit_aps_energy_window(item, fit_lower_ev, fit_upper_ev, fit_points)
                ax.plot(item.energy, item.pes, label=item.name)
                shade_aps_fit_region(ax, fit_lower_ev, fit_upper_ev)
                plot_dat_reference_markers(ax, item)
                plot_aps_fit(ax, item)
            except Exception as exc:
                ax.text(0.02, 0.92, f"Preview fit warning: {exc}", transform=ax.transAxes, fontsize=8)
                break

    ax.set_title("APS Preview with Fitting Region")
    ax.set_xlabel("Energy (eV)")
    ax.set_ylabel(aps_y_label(item_power))
    apply_aps_axis_style(ax)
    show_labeled_legend(ax)


def plot_aps_result_axis(ax, items):
    if not items:
        ax.text(0.5, 0.5, "Run APS analysis to show results.", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return

    for item in items:
        ax.plot(item.energy, item.pes, label=f"{item.name}, HOMO={item.homo:.3g} +/- {item.std_homo:.2g} eV")
        plot_aps_fit(ax, item)

    ax.set_title("APS Analysis Results")
    ax.set_xlabel("Energy (eV)")
    ax.set_ylabel(aps_y_label(items[0].power))
    apply_aps_axis_style(ax)
    show_labeled_legend(ax)


def plot_aps_preview_results(paths, figsize=(8, 6)):
    paths = [path for path in paths if str(path).strip()]
    preview_items = load_aps_files(paths, requested_power=power) if paths else []
    result_items = load_aps_files(paths, requested_power=power) if paths else []
    if result_items:
        analyze_aps_items(result_items)

    fig, (preview_ax, result_ax) = plt.subplots(2, 1, figsize=figsize)
    plot_aps_preview_axis(preview_ax, preview_items)
    plot_aps_result_axis(result_ax, result_items)
    fig.tight_layout(pad=1.2)
    return fig, preview_items, result_items

## Run Preview and Results

In [ ]:
try:
    fig, preview_data, aps_results = plot_aps_preview_results(aps_paths, figsize=(8, 6))
    plt.show()
    for item in aps_results:
        print(f"{item.name}: HOMO={item.homo:.6g} eV, std={item.std_homo:.6g} eV")
except ApsNotebookError as exc:
    print(f"APS notebook error: {exc}")

## Inspect Loaded Objects

In [ ]:
if "preview_data" in globals():
    print(f"Preview objects: {len(preview_data)}")
    for item in preview_data:
        print(item.name, "points=", len(item.energy), "power=", item.power, "analyzed=", item.is_analyzed)